# Speed benchmark tables (LaTeX)

Formats the A100 speed benchmark from `posetail-next/benchmark_speed` (96-frame Point Odyssey clip, `hum0`) into LaTeX,
in the style of the main results table (`tab:main_results`): caption on top, `\rowcolor{rowgray}` striping,
methods down the rows, views (outer) x query points (inner) across; no bold / underline.

* **FPS**: clip frames / median wall time of the full pipeline (decoded frames in RAM -> 3D tracks), 1 warmup + 3 timed runs.
* **Memory**: peak GPU memory allocated by PyTorch (`torch.cuda.max_memory_allocated`), GB, including weights.
* MVTracker (DUSt3R depth) includes DUSt3R depth estimation (the GT-depth variant is left out).
* Point Odyssey has 9 cameras: the 10-view setting duplicates `view2` in the 10th slot.

Needs `\usepackage{booktabs, multirow, colortbl}`, `rowgray` (as in the main table).

In [1]:
import json
from pathlib import Path

RESULTS = Path('/groups/karashchuk/home/karashchukl/projects/posetail/posetail-next/'
               'benchmark_speed/results/a100_mvtracker_20261026_133500')

# rows, top to bottom: (key, display name)
ROWS = [
    ('cotracker',        'CoTracker3 Baseline'),
    ('mvtracker-dust3r', 'MVTracker (DUSt3R depth)~\\cite{rajic2025mvtracker}'),
    ('arrow',            'ARROW~\\cite{fradlin2026arrow}'),
    ('tracktail',        'Tracktail (Ours)'),
]
METHODS = [r[0] for r in ROWS]
VIEWS = [2, 4, 6, 8, 10]
KPTS = [10, 100, 1000]
KPT_LABEL = {10: '10', 100: '100', 1000: '1k'}

res = {}
for m in METHODS:
    for line in open(RESULTS / f'{m}.jsonl'):
        r = json.loads(line)
        res[m, r['views'], r['kpts']] = r   # last record wins
missing = [(m, v, k) for m in METHODS for v in VIEWS for k in KPTS
           if (m, v, k) not in res or res[m, v, k]['status'] != 'ok']
print('missing / failed:', missing)

missing / failed: []


In [2]:
def fmt_fps(x, m):
    # integers, except MVTracker (DUSt3R depth), which is below 1 FPS
    return f'{x:.1f}' if m == 'mvtracker-dust3r' else f'{x:.0f}'

def fmt_mem(x, m):
    return f'{x:.1f}'

def latex_table(key, fmt, caption, label, higher_is_better):
    nk = len(KPTS)
    first = 2   # first numeric column (after Method)
    lines = [
        '\\begin{table*}[t!]',
        '\\centering',
        f'\\caption{{{caption}}}',
        f'\\label{{{label}}}',
        '\\resizebox{\\textwidth}{!}{%',
        '\\begin{tabular}{l ' + ' '.join('c' * nk for _ in VIEWS) + '}',
        '\\toprule',
        'Method',
        '  & ' + '\n  & '.join(f'\\multicolumn{{{nk}}}{{c}}{{{v} views}}' for v in VIEWS) + ' \\\\',
        ''.join(f'\\cmidrule(lr){{{first + i * nk}-{first + (i + 1) * nk - 1}}}' for i in range(len(VIEWS))),
        '\\hfill \\# query points' + ' & ' + ' & '.join(KPT_LABEL[k] for _ in VIEWS for k in KPTS) + ' \\\\',
        '\\midrule',
    ]
    stripe = True
    for row in ROWS:
        m, name = row
        cells = []
        for v in VIEWS:
            for k in KPTS:
                s = fmt(res[m, v, k][key], m)
                cells.append(s)
        prefix = '\\rowcolor{rowgray}\n' if stripe else ''
        lines.append(prefix + name + ' & ' + ' & '.join(cells) + ' \\\\')
        stripe = not stripe
    lines += ['\\bottomrule', '\\end{tabular}%', '}', '\\end{table*}']
    return '\n'.join(lines)

### FPS table

In [3]:
fps_tex = latex_table(
    'fps', fmt_fps,
    caption=('\\textbf{Inference speed} - Frames per second ($\\uparrow$) of the full pipeline on a 96-frame '
             'Point Odyssey clip on one NVIDIA A100 (80\\,GB), for 2--10 views and 10, 100 and 1000 (1k) query points. Timing starts from decoded frames in memory and includes preprocessing, DUSt3R depth estimation '
             'and CoTracker3 triangulation. Point Odyssey has 9 cameras; the 10-view setting repeats one of them.'),
    label='tab:speed_fps', higher_is_better=True)
print(fps_tex)

\begin{table*}[t!]
\centering
\caption{\textbf{Inference speed} - Frames per second ($\uparrow$) of the full pipeline on a 96-frame Point Odyssey clip on one NVIDIA A100 (80\,GB), for 2--10 views and 10, 100 and 1000 (1k) query points. Timing starts from decoded frames in memory and includes preprocessing, DUSt3R depth estimation and CoTracker3 triangulation. Point Odyssey has 9 cameras; the 10-view setting repeats one of them.}
\label{tab:speed_fps}
\resizebox{\textwidth}{!}{%
\begin{tabular}{l ccc ccc ccc ccc ccc}
\toprule
Method
  & \multicolumn{3}{c}{2 views}
  & \multicolumn{3}{c}{4 views}
  & \multicolumn{3}{c}{6 views}
  & \multicolumn{3}{c}{8 views}
  & \multicolumn{3}{c}{10 views} \\
\cmidrule(lr){2-4}\cmidrule(lr){5-7}\cmidrule(lr){8-10}\cmidrule(lr){11-13}\cmidrule(lr){14-16}
\hfill \# query points & 10 & 100 & 1k & 10 & 100 & 1k & 10 & 100 & 1k & 10 & 100 & 1k & 10 & 100 & 1k \\
\midrule
\rowcolor{rowgray}
CoTracker3 Baseline & 134 & 90 & 20 & 67 & 45 & 10 & 45 & 30 & 7 & 3

### Memory table

In [4]:
mem_tex = latex_table(
    'peak_alloc_gb', fmt_mem,
    caption=('\\textbf{GPU memory} - Peak GPU memory allocated (GB, $\\downarrow$), including model weights, '
             'for the runs of Table~\\ref{tab:speed_fps}.'),
    label='tab:speed_memory', higher_is_better=False)
print(mem_tex)

\begin{table*}[t!]
\centering
\caption{\textbf{GPU memory} - Peak GPU memory allocated (GB, $\downarrow$), including model weights, for the runs of Table~\ref{tab:speed_fps}.}
\label{tab:speed_memory}
\resizebox{\textwidth}{!}{%
\begin{tabular}{l ccc ccc ccc ccc ccc}
\toprule
Method
  & \multicolumn{3}{c}{2 views}
  & \multicolumn{3}{c}{4 views}
  & \multicolumn{3}{c}{6 views}
  & \multicolumn{3}{c}{8 views}
  & \multicolumn{3}{c}{10 views} \\
\cmidrule(lr){2-4}\cmidrule(lr){5-7}\cmidrule(lr){8-10}\cmidrule(lr){11-13}\cmidrule(lr){14-16}
\hfill \# query points & 10 & 100 & 1k & 10 & 100 & 1k & 10 & 100 & 1k & 10 & 100 & 1k & 10 & 100 & 1k \\
\midrule
\rowcolor{rowgray}
CoTracker3 Baseline & 8.3 & 8.3 & 9.2 & 8.3 & 8.3 & 9.2 & 8.3 & 8.3 & 9.2 & 8.3 & 8.3 & 9.2 & 8.3 & 8.3 & 9.2 \\
MVTracker (DUSt3R depth)~\cite{rajic2025mvtracker} & 5.0 & 5.0 & 5.0 & 7.7 & 7.7 & 7.7 & 10.4 & 10.4 & 10.5 & 13.2 & 13.2 & 13.2 & 15.9 & 15.9 & 15.9 \\
\rowcolor{rowgray}
ARROW~\cite{fradlin2026arrow} & 7.0 &

In [5]:
# also write the tables next to the notebook
Path('tables').mkdir(exist_ok=True)
Path('tables/speed_fps.tex').write_text(fps_tex + '\n')
Path('tables/speed_memory.tex').write_text(mem_tex + '\n')

1254